# 01 · The PyTorch training loop (drill)

**Style B: blind-code drill.** Write each piece from memory, run the check, delete your code.
Reference: `../solutions/drills/01_training_loop.py`.

**Traces to** `notes/ioai-task-patterns.md` P1, P3, P15 · the loop in every T25/T26 baseline. Gradient descent
and backpropagation are syllabus "Both" items, so section 6 does a forward/backward pass by hand.

Total suggested time: **55 min**.

In [ ]:
import copy, itertools
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

## 1. Data loaders ⏱ 6 min

FashionMNIST from `torchvision.datasets` (root `"data"`):
- `train_loader`: the 60k training images, batch 128, **shuffled**.
- `test_loader`: the 10k test images, batch 512, not shuffled.

Batches must be `x: float32 (B, 1, 28, 28)` in [0, 1] and `y: int64 (B,)`.

In [ ]:
x, y = next(iter(train_loader))
assert x.shape == (128, 1, 28, 28) and x.dtype == torch.float32 and 0 <= x.min() and x.max() <= 1
assert y.dtype == torch.int64 and y.shape == (128,)
assert len(train_loader.dataset) == 60000 and len(test_loader.dataset) == 10000
y1 = next(iter(train_loader))[1]; y2 = next(iter(train_loader))[1]
assert not torch.equal(y1, y2), "train_loader is not shuffled"
assert torch.equal(next(iter(test_loader))[1], next(iter(test_loader))[1]), "test_loader must not shuffle"
print("section 1 ok")

## 2. A small CNN ⏱ 7 min

`make_model() -> nn.Module` mapping `(B, 1, 28, 28)` to **raw logits** `(B, 10)`, with at most 150k
parameters. Use conv + pooling (+ BatchNorm if you like).

In [ ]:
m = make_model()
n_params = sum(p.numel() for p in m.parameters())
out = m(torch.randn(4, 1, 28, 28))
assert out.shape == (4, 10) and n_params <= 150_000, (out.shape, n_params)
assert not torch.allclose(out.softmax(1).sum(1), out.sum(1)), "looks like you return probabilities"
print(f"section 2 ok ({n_params} params)")

## 3. One epoch ⏱ 6 min

`train_one_epoch(model, loader, optimizer, loss_fn) -> float`: puts the model in train mode, moves the batches to
`device`, does the optimizer step, and returns the **mean loss per sample** over the epoch.

In [ ]:
torch.manual_seed(1)
m = make_model().to(device)
opt = torch.optim.Adam(m.parameters(), lr=1e-3)
small = DataLoader(Subset(train_loader.dataset, range(3000)), batch_size=64, shuffle=True)
l1 = train_one_epoch(m, small, opt, nn.CrossEntropyLoss())
l2 = train_one_epoch(m, small, opt, nn.CrossEntropyLoss())
assert isinstance(l1, float) and l2 < l1 and m.training
print(f"section 3 ok ({l1:.3f} -> {l2:.3f})")

## 4. Evaluation ⏱ 5 min

`evaluate(model, loader) -> float` accuracy. It must leave the model in eval mode and must not build autograd graphs.

In [ ]:
m.train()
acc = evaluate(m, test_loader)
assert not m.training
with torch.no_grad():
    correct = sum((m(xb.to(device)).argmax(1).cpu() == yb).sum().item() for xb, yb in test_loader)
ref = correct / len(test_loader.dataset)
assert abs(acc - ref) < 1e-6, (acc, ref)
print(f"section 4 ok (accuracy {acc:.4f})")

## 5. `fit` with a schedule and best-epoch checkpoint ⏱ 12 min

`fit(model, train_loader, val_loader, epochs, lr) -> best_state_dict`: AdamW + any LR scheduler. Evaluate on
`val_loader` every epoch and return a **copy** of the weights from the best epoch. The check fits 4 epochs on
20k images and needs ≥ 0.86 test accuracy from the returned weights.

In [ ]:
torch.manual_seed(2)
m = make_model().to(device)
tr = DataLoader(Subset(train_loader.dataset, range(20000)), batch_size=128, shuffle=True)
va = DataLoader(Subset(train_loader.dataset, range(55000, 60000)), batch_size=512)
state = fit(m, tr, va, epochs=4, lr=3e-3)
k = next(iter(state))
assert state[k].data_ptr() != m.state_dict()[k].data_ptr(), "return a copy, not the live state_dict"
fresh = make_model().to(device); fresh.load_state_dict(state)
acc = evaluate(fresh, test_loader)
assert acc >= 0.86, acc
print(f"section 5 ok (test accuracy {acc:.4f})")

## 6. Backprop by hand ⏱ 12 min

`manual_mlp_grads(X, y, W1, b1, W2, b2) -> (loss, dW1, db1, dW2, db2)` for
`h = relu(X @ W1 + b1)`, `logits = h @ W2 + b2`, `loss = mean cross-entropy(logits, y)`.
Use plain tensor ops only (no `.backward()`, no `torch.autograd`). The check compares against autograd.

In [ ]:
g = torch.Generator().manual_seed(0)
X = torch.randn(16, 5, generator=g); yy = torch.randint(0, 3, (16,), generator=g)
params = [torch.randn(5, 7, generator=g), torch.randn(7, generator=g), torch.randn(7, 3, generator=g), torch.randn(3, generator=g)]
loss, *grads = manual_mlp_grads(X, yy, *[p.clone() for p in params])
ps = [p.clone().requires_grad_() for p in params]
ref = F.cross_entropy(F.relu(X @ ps[0] + ps[1]) @ ps[2] + ps[3], yy)
ref.backward()
assert torch.isclose(torch.as_tensor(loss), ref.detach(), atol=1e-5)
for mine, p in zip(grads, ps):
    assert torch.allclose(mine, p.grad, atol=1e-5), "gradient mismatch"
print("section 6 ok")